In [8]:
"""
fig1_correlation_heatmap.py
----------------------------
Generates Figure 1: Pairwise Return Correlation Matrix for the 22-stock
NGX universe (2015-2025), sorted by sector with banking cluster visible.

Input:  correlation_matrix.csv  (22x22 Pearson correlation matrix)
Output: fig1_correlation_heatmap.pdf  (for LaTeX inclusion)
        fig1_correlation_heatmap.png  (for preview)

Requirements:
    pip install matplotlib seaborn numpy pandas
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

In [9]:
# ── 1. Load data ───────────────────────────────────────────────────────────────
corr = pd.read_csv('C:/Users/hp/Documents/Other_documents/PhD_files/PhD Thesis/phd-research-portfolio-optimization/results/correlation_matrix.csv', index_col=0)

In [10]:
# ── 2. Sector classification ───────────────────────────────────────────────────
sector = {
    'DANGCEM':    'Indus',
    'WAPCO':      'Indus',
    'DANGSUG':    'Consumer Goods',
    'GUINNES':    'Consumer Goods',
    'INTBREW':    'Consumer Goods',
    'NB':         'Consumer Goods',
    'NESTLE':     'Consumer Goods',
    'ETI':        'Financial Svcs',
    'FCMB':       'Financial Svcs',
    'FIDELIT':    'Financial Svcs',
    'FIRSTHOLD':  'Financial Svcs',
    'GTCO':       'Financial Svcs',
    'IBTC':       'Financial Svcs',
    'STERLINGNG': 'Financial Svcs',
    'UBA':        'Financial Svcs',
    'UCAP':       'Financial Svcs',
    'ZENITHB':    'Financial Svcs',
    'OKOMUOI':    'Agric',
    'PRESCO':     'Agric',
    'SEPLAT':     'O&Gas',
    'TOTAL':      'O&Gas',
    'TRANSCO':    'Conglo',
}

sector_order = [
    'Financial Svcs', 'Consumer Goods', 'O&Gas',
    'Agric', 'Indus', 'Conglo'
]

sector_colors = {
    'Financial Svcs': '#1f4e79',
    'Consumer Goods': '#2e7d32',
    'O&Gas':      '#b45309',
    'Agric':    '#6a1070',
    'Indus':    '#1a5276',
    'Conglo':   '#7b241c',
}

In [11]:
# ── 3. Sort tickers by sector then alphabetically ─────────────────────────────
tickers_sorted = sorted(
    corr.index,
    key=lambda t: (sector_order.index(sector.get(t, 'Conglo')), t)
)
corr_sorted = corr.loc[tickers_sorted, tickers_sorted]

In [12]:
# ── 4. Build figure ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8.5))

# Custom diverging colormap: deep red → white → deep blue
cmap = LinearSegmentedColormap.from_list(
    'ngx_corr',
    ['#d32f2f', '#ef9a9a', '#ffffff', '#90caf9', '#0d47a1'],
    N=256
)

mask = np.eye(len(corr_sorted), dtype=bool)   # mask diagonal
mat  = corr_sorted.values.astype(float)
n    = len(tickers_sorted)

im = ax.imshow(
    np.ma.masked_where(mask, mat),
    cmap=cmap, vmin=-0.15, vmax=0.60,
    aspect='auto', interpolation='nearest'
)

# ── 5. Tick labels ────────────────────────────────────────────────────────────
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(tickers_sorted, rotation=45, ha='right',
                   fontsize=8.5, fontfamily='DejaVu Sans')
ax.set_yticklabels(tickers_sorted, fontsize=8.5, fontfamily='DejaVu Sans')

# ── 6. Grey fill on diagonal cells ───────────────────────────────────────────
for i in range(n):
    ax.add_patch(plt.Rectangle(
        (i - 0.5, i - 0.5), 1, 1,
        fc='#d0d0d0', ec='none', zorder=2
    ))

# ── 7. Annotate cells where ρ > 0.25 ─────────────────────────────────────────
for i in range(n):
    for j in range(n):
        if i != j and mat[i, j] > 0.25:
            ax.text(j, i, f'{mat[i, j]:.2f}',
                    ha='center', va='center', fontsize=6.2,
                    color='white', fontweight='bold', zorder=3)

# ── 8. Sector bracket lines (left and top) ────────────────────────────────────
current_sector = None
start_idx = 0
sector_boundaries = []

for idx, t in enumerate(tickers_sorted):
    s = sector.get(t)
    if s != current_sector:
        if current_sector is not None:
            sector_boundaries.append((current_sector, start_idx, idx - 1))
        current_sector = s
        start_idx = idx
sector_boundaries.append((current_sector, start_idx, n - 1))

for sec_name, s_start, s_end in sector_boundaries:
    col = sector_colors[sec_name]
    mid = (s_start + s_end) / 2

    # Left bar
    ax.plot([-1.5, -1.5], [s_start - 0.45, s_end + 0.45],
            color=col, lw=4, solid_capstyle='butt',
            transform=ax.transData, clip_on=False, zorder=5)
    ax.text(-2.1, mid, sec_name,
            ha='left', va='center', fontsize=7.5,
            color=col, fontweight='bold',
            fontfamily='DejaVu Sans',
            transform=ax.transData, clip_on=False, rotation=90)

    # Top bar
    ax.plot([s_start - 0.45, s_end + 0.45], [-1.5, -1.5],
            color=col, lw=4, solid_capstyle='butt',
            transform=ax.transData, clip_on=False, zorder=5)

# ── 9. Colour bar ─────────────────────────────────────────────────────────────
cbar = fig.colorbar(im, ax=ax, fraction=0.032, pad=0.02)
cbar.ax.tick_params(labelsize=9)
cbar.set_label('Pearson Correlation $\\rho$', fontsize=10)

# ── 10. Grid lines ────────────────────────────────────────────────────────────
for i in range(n + 1):
    ax.axhline(i - 0.5, color='white', lw=0.35, zorder=1)
    ax.axvline(i - 0.5, color='white', lw=0.35, zorder=1)

# ── 11. Title ─────────────────────────────────────────────────────────────────
ax.set_title(
    'Pairwise Return Correlation Matrix for 22 NGX Stocks (2015–2025)\n'
    r'Mean $\bar{\rho} = 0.109$; Max $\rho = 0.581$ (GTCO–ZENITHB); '
    '12 of 231 pairs exceed 0.30',
    fontsize=10.5, pad=14, fontfamily='DejaVu Sans'
)

# ── 12. Save ──────────────────────────────────────────────────────────────────
plt.tight_layout(rect=[0.13, 0, 1, 1])
fig.savefig('fig1_correlation_heatmap.pdf', dpi=300,
            bbox_inches='tight', format='pdf')
fig.savefig('fig1_correlation_heatmap.png', dpi=300,
            bbox_inches='tight')
plt.close()
print("Figure 1 saved: fig1_correlation_heatmap.pdf / .png")

Figure 1 saved: fig1_correlation_heatmap.pdf / .png
